# Imports

In [1]:
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from tqdm import tqdm
import plotly.express as px

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 300)

In [3]:
file_path = 'E:/ML/datasets/mahjong/data/2019/block_5000.parquet'

In [50]:
states = pd.read_parquet(file_path)

In [51]:
states.columns = states.columns.map(
    str
)

In [6]:
single_round = states.loc[(states['511'] == '009379d9') & (states['32'] == '0')]

# Round 2 Vec

In [7]:
def get_perspective_position(metadata_vector):
    n = metadata_vector.iloc[2]
    arr = np.asarray(metadata_vector[6:10])
    if np.all(arr == arr[0]):
        return 'tie'
    position = int(np.sum(arr < arr[n]))

    position_dict = {0:'1st',
                     1:'2nd',
                     2:'3rd',
                     3:'4th'}
    return position_dict[position]

In [8]:
def drop_nth(s, n):
    mask = np.ones(len(s), dtype=bool)
    mask[n] = False
    return s.iloc[mask]

In [9]:
def id_to_tile(id):
    return id %34

In [11]:
def normalize_seats(seats_vector):
    seats_vector =  seats_vector.astype(int)
    persp_seat = seats_vector.iloc[-1]
    persp_dict = {persp_seat: "P",
                  (persp_seat+1)%4: "R",
                  (persp_seat-1)%4: "L",
                  (persp_seat+2)%4: "A"}
    return seats_vector.replace(persp_dict)

In [10]:
mahjong_tiles = [
    # Manzu (Characters)
    "1m", "2m", "3m", "4m", "5m", "6m", "7m", "8m", "9m",
    
    # Souzu (Bamboos)
    "1s", "2s", "3s", "4s", "5s", "6s", "7s", "8s", "9s",
    
    # Pinzu (Circles)
    "1p", "2p", "3p", "4p", "5p", "6p", "7p", "8p", "9p",
    
    # Winds
    "East", "South", "West", "North",
    
    # Dragons (Colors)
    "White", "Green", "Red"
]

In [52]:
def round_2_vec(rnd):
    # Metadata section
    metadata_vector = rnd.iloc[34][:68].astype(int)
    
    lookup = np.array(['E','S','W','N'])
    round_wind = lookup[metadata_vector.iloc[0]]

    seat_wind = lookup[(metadata_vector.iloc[2] - metadata_vector.iloc[1]) % 4]

    if (metadata_vector.iloc[1] == metadata_vector.iloc[2]):
        is_dealer = 'dealer'
    else:
        is_dealer = 'nondealer'

    if metadata_vector.iloc[6] < 20:
        wall = 'WU20'
    else:
        wall = 'WOoA20'

    relative_score = get_perspective_position(metadata_vector)

    if metadata_vector[10:14].iloc[metadata_vector.iloc[2]]:
        did_riichi = 'did_riichi'
    else:
        did_riichi = 'no_riichi'
        
    other_riichis = f'{sum(drop_nth(metadata_vector.iloc[10:14],metadata_vector.iloc[2]))}_OR'

    dora_series = metadata_vector.iloc[34:68]

    dora = id_to_tile(int(dora_series.loc[dora_series==1].index[0]))

    dora = f'D{mahjong_tiles[dora]}'

    # Discard Section
    discards = single_round['510'][:35].astype(int)
    discard_vector = pd.Series(np.take(mahjong_tiles, discards.values), index=discards.index)
    discard_vector = normalize_seats(single_round['2'][:35]) + discard_vector
    
    return np.concatenate(([round_wind], [seat_wind], [wall], [relative_score], [did_riichi], [other_riichis], [dora], discard_vector))

In [53]:
round_2_vec(single_round)

array(['E', 'W', 'WOoA20', 'tie', 'no_riichi', '0_OR', 'D5m', 'ASouth',
       'LSouth', 'P8p', 'RSouth', 'AWhite', 'L9s', 'PWhite', 'R1p', 'A9s',
       'LRed', 'PEast', 'REast', 'A7p', 'LWest', 'P1m', 'R8p', 'A2p',
       'L9m', 'P2m', 'A4p', 'L6s', 'PWest', 'R8s', 'A3p', 'L1m', 'P5p',
       'R7p', 'A5p', 'L1m', 'P7s', 'A6s', 'L3s', 'A4s', 'LWhite', 'P7p'],
      dtype=object)

# Get all files

# Identify Chiitoi States

In [55]:
is_chiitoi = (states[states.columns[68:102]].astype(int) == 2).sum(axis=1) == 6

In [56]:
is_chiitoi.value_counts()

False    2405241
True        7677
Name: count, dtype: int64

In [57]:
states['is_chiitoi'] = is_chiitoi

In [58]:
keys = ['32','511','is_chiitoi']

In [59]:
cumsums = states.groupby(keys, sort=False)['is_chiitoi'].cumsum()

In [61]:
mask = (cumsums == 1)

In [62]:
result = states[mask]

In [60]:
cumsums.value_counts()

is_chiitoi
0     2405241
1        3375
2        1608
3        1013
4         637
5         413
6         243
7         153
8          99
9          59
10         34
11         19
12         12
13          8
14          3
15          1
Name: count, dtype: int64

In [63]:
result

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,333,334,335,336,337,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383,384,385,386,387,388,389,390,391,392,393,394,395,396,397,398,399,400,401,402,403,404,405,406,407,408,409,410,411,412,413,414,415,416,417,418,419,420,421,422,423,424,425,426,427,428,429,430,431,432,433,434,435,436,437,438,439,440,441,442,443,444,445,446,447,448,449,450,451,452,453,454,455,456,457,458,459,460,461,462,463,464,465,466,467,468,469,470,471,472,473,474,475,476,477,478,479,480,481,482,483,484,485,486,487,488,489,490,491,492,493,494,495,496,497,498,499,500,501,502,503,504,505,506,507,508,509,510,511,is_chiitoi
1449,0,2,1,2,1,27,31,22,21,24,0,0,0,0,-128,-128,-128,-128,-128,-128,-128,-128,-128,-128,-128,-128,-128,-128,-128,-128,-128,-128,6,-40,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,2,0,0,2,2,0,0,0,0,0,0,0,0,0,2,0,0,0,2,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,2,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,1,1,1,2,1,1,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,2,1,0,0,0,0,0,0,0,0,1,0,1,1,1,1,0,2,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,1,1,0,1,0,1,0,1,0,1,1,0,0,2,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,1,1,1,2,1,1,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,2,1,0,0,0,0,0,0,0,0,1,0,1,1,1,1,0,2,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,1,1,0,1,33,4f5e8a98,True
2299,1,0,2,0,0,36,31,23,9,37,0,0,0,0,-128,-128,-128,-128,-128,-128,-128,-128,-128,-128,-128,-128,-128,-128,-128,-128,-128,-128,7,-56,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,2,2,2,0,0,0,0,0,0,0,0,0,2,2,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,2,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0,1,1,0,1,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,1,0,0,0,1,1,0,0,0,0,0,0,1,0,0,0,0,1,1,0,0,0,0,2,0,0,1,0,1,0,0,0,0,0,0,0,0,0,2,0,0,1,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,1,2,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0,1,1,

In [42]:
states.shape

(2412918, 513)